In [57]:
from langchain.document_loaders import PyPDFLoader,DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [58]:
def load_pdf_file(data):
    loader=DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    document = loader.load()

    return document

In [59]:
extracted_data=load_pdf_file("../data")
print(extracted_data[10])

page_content='Rhonda Cloos, R.N.
Medical Writer
Austin, TX
Gloria Cooksey, C.N.E
Medical Writer
Sacramento, CA
Amy Cooper, M.A., M.S.I.
Medical Writer
Vermillion, SD
David A. Cramer, M.D.
Medical Writer
Chicago, IL
Esther Csapo Rastega, R.N., B.S.N.
Medical Writer
Holbrook, MA
Arnold Cua, M.D.
Physician
Brooklyn, NY
Tish Davidson, A.M.
Medical Writer
Fremont, California
Dominic De Bellis, Ph.D.
Medical Writer/Editor
Mahopac, NY
Lori De Milto
Medical Writer
Sicklerville, NJ
Robert S. Dinsmoor
Medical Writer
South Hamilton, MA
Stephanie Dionne, B.S.
Medical Writer
Ann Arbor, MI
Martin W. Dodge, Ph.D.
Technical Writer/Editor
Centinela Hospital and Medical
Center
Inglewood, CA
David Doermann
Medical Writer
Salt Lake City, UT
Stefanie B. N. Dugan, M.S.
Genetic Counselor
Milwaukee, WI
Doug Dupler, M.A.
Science Writer
Boulder, CO
Julie A. Gelderloos
Biomedical Writer
Playa del Rey, CA
Gary Gilles, M.A.
Medical Writer
Wauconda, IL
Harry W. Golden
Medical Writer
Shoreline Medical Writers
Old Ly

In [60]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs:List[Document]) -> List[Document]:
    """
        Given a list of Document objects, return a new list of Document objects
        containing only 'source' in metadata and the original page_content.
    """

    minimal_docs:List[Document]=[]

    for doc in docs:
        src=doc.metadata.get("source")
        page_content = doc.page_content
        minimal_docs.append(
            Document(
                page_content=page_content,
                metadata={"source":src}
            )
        )

    return minimal_docs

In [61]:
minimzied_pdf=filter_to_minimal_docs(extracted_data)

print(minimzied_pdf[10])

page_content='Rhonda Cloos, R.N.
Medical Writer
Austin, TX
Gloria Cooksey, C.N.E
Medical Writer
Sacramento, CA
Amy Cooper, M.A., M.S.I.
Medical Writer
Vermillion, SD
David A. Cramer, M.D.
Medical Writer
Chicago, IL
Esther Csapo Rastega, R.N., B.S.N.
Medical Writer
Holbrook, MA
Arnold Cua, M.D.
Physician
Brooklyn, NY
Tish Davidson, A.M.
Medical Writer
Fremont, California
Dominic De Bellis, Ph.D.
Medical Writer/Editor
Mahopac, NY
Lori De Milto
Medical Writer
Sicklerville, NJ
Robert S. Dinsmoor
Medical Writer
South Hamilton, MA
Stephanie Dionne, B.S.
Medical Writer
Ann Arbor, MI
Martin W. Dodge, Ph.D.
Technical Writer/Editor
Centinela Hospital and Medical
Center
Inglewood, CA
David Doermann
Medical Writer
Salt Lake City, UT
Stefanie B. N. Dugan, M.S.
Genetic Counselor
Milwaukee, WI
Doug Dupler, M.A.
Science Writer
Boulder, CO
Julie A. Gelderloos
Biomedical Writer
Playa del Rey, CA
Gary Gilles, M.A.
Medical Writer
Wauconda, IL
Harry W. Golden
Medical Writer
Shoreline Medical Writers
Old Ly

In [62]:
def text_split(minimzied_pdf):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )

    splitted_pdf=text_splitter.split_documents(minimzied_pdf)
    return splitted_pdf

In [63]:
splitted_pdf=text_split(minimzied_pdf)

print(f"Number of chunk is {len(splitted_pdf)}")

Number of chunk is 5859


In [64]:
from langchain.embeddings import HuggingFaceBgeEmbeddings

def convert_into_embadding():
    """
        Download and return huggingface embedding model
    """

    model_name="sentence-transformers/all-MiniLM-L6-v2"
    embedding = HuggingFaceBgeEmbeddings(
        model_name=model_name
    )

    return embedding

embedding=convert_into_embadding()



In [65]:
vector=embedding.embed_query("My name is Aryan")
print(len(vector))

384


In [66]:
from dotenv import load_dotenv
load_dotenv()
import os

In [67]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"]=PINECONE_API_KEY
os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY

In [68]:
from pinecone import Pinecone

pc=Pinecone(api_key=PINECONE_API_KEY)

pc

In [69]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot-genai"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws",region="us-east-1")
    )

index=pc.Index(index_name)

In [70]:
from langchain_pinecone import PineconeVectorStore

# docsearch = PineconeVectorStore.from_documents(
#     documents=splitted_pdf,
#     embedding=embedding,
#     index_name=index_name
# )


In [71]:
# Load Exesting Index
from langchain_pinecone import PineconeVectorStore

docsearch=PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

#ADD MORE DATA TO EXESTING PINECONE DATABASE



In [72]:
dummy_doc=Document(
    page_content="My Name is Aryan Tiwari,Aryan Tiwari is a student",
    metadata={"source":"LinkedIN"}
)

docsearch.add_documents(documents=[dummy_doc])

['a1329a6f-320f-4f44-aff6-77a908d3aea1']

#Extracting data from Pinecone Index


In [73]:
retriever = docsearch.as_retriever(search_type="similarity",search_kwargs={"k":3})

In [74]:
response=retriever.invoke("How many years, on average, after the initial episode does the first recurrence of Bell’s palsy occur?")
print(response[0])

page_content='experienced Bell’s palsy will have a recurrence. On aver-
age, the first recurrence happens 9.8 years after the first
episode; the second, 6.7 years later. One recurrence is
very infrequent, and a second is extremely rare.
Causes and symptoms
The symptoms of Bell’s palsy arise from an inflam-
mation or swelling of the seventh cranial nerve, other-
wise called the facial nerve. Both sides of the face have a
facial nerve which controls the muscles on that side of' metadata={'source': '..\\data\\Medical_book.pdf'}


# Connecting with LLM


In [75]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

In [76]:
model=ChatGroq(model="openai/gpt-oss-20b",temperature=0)
result=model.invoke("Which country playes todays cricket match?")
print(result.content)

I’m sorry, but I don’t have real‑time access to today’s cricket fixtures. To find out which teams are playing on 12 Oct 2026, you can check a reliable cricket‑scheduling site such as:

- **ESPNcricinfo** (live scores & fixtures)
- **ICC official website** (international series calendar)
- **Cricket.com.au** or **Cricbuzz** (regional schedules)

Those sites will list the matches, venues, and start times for the day. If you let me know a specific tournament or region you’re interested in, I can give you a general idea of what’s usually happening around that time.


# Chain Creation


In [77]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate



In [78]:
system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt=ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","{input}")
    ]
)

In [79]:
question_answer_chain=create_stuff_documents_chain(model,prompt)
rag_chain=create_retrieval_chain(retriever,question_answer_chain)

In [111]:
#COMPARING WITH LLM RESPONSE
result=model.invoke("In the MDMDA survey, what did bipolar patients report about getting a correct diagnosis?in one line")
print(result.content)

Bipolar patients reported that most struggled to get an accurate diagnosis, with many experiencing misdiagnosis or delayed recognition.


In [113]:
# RESPONSE GIVEN BY OUR CHATBOT
response=rag_chain.invoke({"input": "Which organism causes amebiasis, and how many new cases occur each year?"})
print(response["answer"])

Amebiasis is caused by the protozoan parasite *Entamoeba histolytica*. It is estimated that about 500 million new cases occur worldwide each year.


In [ ]:
How many people die of amebiasis each year, according to the book?
What percentage of the general US population develops amebiasis in a given year?
How common is bladder cancer in the US, and what were the ACS estimates for 1998?
Which organism causes amebiasis, and how many new cases occur each year?
In the MDMDA survey, what did bipolar patients report about getting a correct diagnosis?


Object `book` not found.
Object `year` not found.
